# 15 · Random forest

Bagged trees with random feature subsets. The forest stays unscaled and uses class-balanced weights. Frozen train and validation arrays only; no test data.


In [1]:
from pathlib import Path
import sys,json,time
import joblib,numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd();sys.path.insert(0,str(ROOT/'src'))
from dronedet.config import load_project_config
from dronedet.features import load_tabular
from dronedet.metrics import classification_metrics,select_f1_threshold,miss_rate_at_false_alarm_rate
cfg=load_project_config(ROOT/'config.yaml');P=cfg['P']


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Inputs and prewritten tuning plan

The default has 500 trees, `max_features="sqrt"`, no depth cap, leaf size 1 and `class_weight="balanced"`. One-factor alternatives create eight settings, each with three seeds. Accepted changes are combined and checked with three more fits. The forest size curve is diagnostic only.


In [2]:
tr=load_tabular(P,'train');va=load_tabular(P,'val');Xtr=np.asarray(tr['X'],dtype=np.float32);ytr=np.asarray(tr['y'],dtype=np.int8);Xva=np.asarray(va['X'],dtype=np.float32);yva=np.asarray(va['y'],dtype=np.int8);names=np.asarray(tr['feature_names']).astype(str)
assert Xtr.shape==(7607,84) and Xva.shape==(1510,84)
assert np.array_equal(tr['clip_id'],pd.read_csv(P['splits']/'train.csv',keep_default_na=False).clip_id) and np.array_equal(va['clip_id'],pd.read_csv(P['splits']/'val.csv',keep_default_na=False).clip_id)
seeds=[int(cfg['seed']),int(cfg['seed'])+1,int(cfg['seed'])+2];default={'max_features':'sqrt','min_samples_leaf':1,'max_depth':None,'class_weight':'balanced'}
configs=[default.copy()]+[dict(default,max_features=v) for v in ('log2',.3)]+[dict(default,min_samples_leaf=v) for v in (3,5)]+[dict(default,max_depth=v) for v in (12,24)]+[dict(default,class_weight='balanced_subsample')]
def fit_eval(hp,seed,n_estimators=500):
 m=RandomForestClassifier(n_estimators=n_estimators,**hp,n_jobs=-1,random_state=seed);m.fit(Xtr,ytr);sc=m.predict_proba(Xva)[:,1];th=select_f1_threshold(yva,sc);met=classification_metrics(yva,sc,threshold=th);return m,{'threshold':float(th),'val_f1':met['f1'],'val_roc_auc':met['roc_auc'],'far':met['false_alarm_rate'],'miss':met['miss_rate']}
rows=[];models={};t0=time.perf_counter()
for ci,hp in enumerate(configs):
 for seed in seeds:
  m,met=fit_eval(hp,seed);rows.append({'config_id':ci,'seed':seed,**hp,**met});models[(ci,seed)]=m
base=float(pd.DataFrame(rows).query('config_id==0').val_f1.mean());summ=pd.DataFrame(rows).groupby('config_id',as_index=False).agg(mean_f1=('val_f1','mean'),std_f1=('val_f1','std'),mean_auc=('val_roc_auc','mean'));summ['delta']=summ.mean_f1-base;summ['accepted']=(summ.config_id!=0)&(summ.delta>=.01)&(summ.delta>float(summ.loc[summ.config_id==0,'std_f1'].iloc[0]))
accepted=summ[summ.accepted].config_id.tolist();combo_id=None
if accepted:
 combo=default.copy()
 for ci in accepted: combo.update({k:v for k,v in configs[ci].items() if v!=default[k]})
 combo_id=len(configs);configs.append(combo)
 for seed in seeds:
  m,met=fit_eval(combo,seed);rows.append({'config_id':combo_id,'seed':seed,**combo,**met});models[(combo_id,seed)]=m
 frame=pd.DataFrame(rows);combo_mean=frame[frame.config_id==combo_id].val_f1.mean();combo_std=frame[frame.config_id==combo_id].val_f1.std();best_single=float(summ.mean_f1.max())
 if combo_mean<best_single+.01 or combo_mean<=float(summ.loc[summ.mean_f1.idxmax(),'std_f1']): combo_id=None
frame=pd.DataFrame(rows);valid_ids=[0]+[int(i) for i in accepted if i!=combo_id]+([combo_id] if combo_id is not None else [])
summary=frame[frame.config_id.isin(valid_ids)].groupby('config_id',as_index=False).agg(mean_f1=('val_f1','mean'),std_f1=('val_f1','std'),mean_auc=('val_roc_auc','mean'))
chosen_id=int(summary.sort_values(['mean_f1','config_id'],ascending=[False,True]).iloc[0].config_id)
seed_best=frame[frame.config_id==chosen_id].sort_values(['val_f1','seed'],ascending=[False,True]).iloc[0];chosen_seed=int(seed_best.seed);model=models[(chosen_id,chosen_seed)]
score=model.predict_proba(Xva)[:,1];threshold=select_f1_threshold(yva,score);metrics=classification_metrics(yva,score,threshold=threshold)
print('Initial configs:',len(configs)-(1 if combo_id is not None else 0),'fits:',len(frame),'accepted singles:',accepted,'combo:',combo_id,'frozen:',configs[chosen_id], 'F1:',metrics['f1'])


Initial configs: 8 fits: 24 accepted singles: [] combo: None frozen: {'max_features': 'sqrt', 'min_samples_leaf': 1, 'max_depth': None, 'class_weight': 'balanced'} F1: 0.8472622478386167


## Saturation, importance and freeze

Train the default forest with `warm_start` and inspect validation performance at 50–500 trees. Permutation importance uses validation data only and is descriptive; it does not tune the model.


In [3]:
curve=[];warm=RandomForestClassifier(n_estimators=50,**default,n_jobs=-1,random_state=seeds[0],warm_start=True)
for n in (50,100,200,300,500):
 warm.set_params(n_estimators=n);warm.fit(Xtr,ytr);curve.append({'trees':n,'val_auc':float(__import__('sklearn').metrics.roc_auc_score(yva,warm.predict_proba(Xva)[:,1]))})
perm=permutation_importance(model,Xva,yva,scoring='roc_auc',n_repeats=3,random_state=seeds[0],n_jobs=-1)
miss1,thr1=miss_rate_at_false_alarm_rate(yva,score,max_false_alarm_rate=.01);miss5,thr5=miss_rate_at_false_alarm_rate(yva,score,max_false_alarm_rate=.05)
FIG=ROOT/'results'/'figures';TABLES=ROOT/'results'/'tables';MODELS=ROOT/'results'/'models';CARDS=ROOT/'results'/'cards'
for d in (FIG,TABLES,MODELS,CARDS):d.mkdir(parents=True,exist_ok=True)
frame.to_csv(TABLES/'15_random_forest_seed_metrics.csv',index=False);pd.DataFrame(curve).to_csv(TABLES/'15_random_forest_saturation.csv',index=False)
imp=pd.DataFrame({'feature':names,'importance_mean':perm.importances_mean,'importance_std':perm.importances_std}).sort_values('importance_mean',ascending=False);imp.to_csv(TABLES/'15_random_forest_permutation_importance.csv',index=False)
fig,ax=plt.subplots(figsize=(7,5));ax.plot([x['trees'] for x in curve],[x['val_auc'] for x in curve],'o-');ax.set(xlabel='Trees',ylabel='Validation ROC-AUC',title='Forest saturation');fig.tight_layout();fig.savefig(FIG/'15_random_forest_saturation.png',dpi=160);plt.close(fig)
card={'model':'random_forest','representation':'84 tabular features','best_hyperparameters':configs[chosen_id],'val_metrics':{**metrics,'threshold':float(threshold),'miss_rate_at_far_1pct':float(miss1),'threshold_at_far_1pct':float(thr1),'miss_rate_at_far_5pct':float(miss5),'threshold_at_far_5pct':float(thr5)},'seeds':seeds,'selected_seed':chosen_seed,'train_time_s':float(time.perf_counter()-t0),'diagnostics':{'tree_count':500,'total_nodes':int(sum(est.tree_.node_count for est in model.estimators_)),'accepted_single_variants':accepted,'combination_retained':combo_id is not None},'decision_log_ids':['D044','D045']}
fh=json.loads((P['spec']/'frontend_hash.json').read_text(encoding='utf-8'))['params_hash'];joblib.dump({'model':model,'feature_names':names.tolist(),'threshold':float(threshold),'threshold_rule':'validation_f1_max_ties_higher','front_end_hash':fh,'hyperparameters':card['best_hyperparameters']},MODELS/'15_random_forest.joblib');(CARDS/'15_random_forest.json').write_text(json.dumps(card,indent=2,default=float),encoding='utf-8');print('Validation:',metrics)


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\sklearn\ensemble\_forest.py:843: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\sklearn\ensemble\_forest.py:843: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\sklearn\ensemble\_forest.py:843: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\sklearn\ensemble\_forest.py:843: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\sklearn\ensemble\_forest.py:843: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(


Validation: {'accuracy': 0.9649006622516556, 'precision': 0.9074074074074074, 'recall': 0.7945945945945946, 'f1': 0.8472622478386167, 'roc_auc': 0.9636308006119327, 'false_alarm_rate': 0.011320754716981131, 'miss_rate': 0.20540540540540542, 'confusion_matrix': [[1310, 15], [38, 147]]}
